# Notebook 02 — Classification Analysis

Behavioural feature exploration + genotype classification. Pool multiple tracking runs (same day / genotype layout) for more flies per genotype.

## Stages
1. Configuration (`RUN_DIRS`, `CV_MODE`, auto-incremented `FIGURES_DIR`)
2. Load `compact_tracks.csv` per run, map genotypes, concat with run-scoped `compact_id`
3. Extract frame-level behavioural features
4. Aggregate to per-fly features (+ `run` column)
5. Feature list + **per-run sanity** box plots (catch video/batch offsets)
6. Exploratory plots (by genotype, WT vs mutant)
7. Classification (LDA, Logistic, SVC): **stratified** CV or **group** CV (leave-one-video-out)
8. Optional: single `24DPE.html` — pooled + per-trial report (same plots + classifiers unpooled)

**Edit the configuration cell** (`run_multiple_videos`, `RUN_DIRS` / tag, `CV_MODE`).

In [ ]:
import sys
sys.path.insert(0, '..')  # so 'src' is importable from notebooks/

import os
import pandas as pd
import plotly.express as px

from src.classification import (
    map_vial_to_genotype,
    run_classifier,
    plot_by_genotype,
    plot_wt_vs_mutant,
    write_classification_report_site,
)
from src.features import extract_behavioral_features, aggregate_per_fly_features
from src.latent_space import run_latent_space_analysis

## 1 — Configuration

In [ ]:
# ---- EDIT THIS ----
run_multiple_videos = True
CV_MODE = "group"  # "stratified" — pooled flies | "group" — leave-one-video-out (GroupKFold)
RUN_LATENT_SPACE = True
WRITE_REPORT_SITE = True
assert CV_MODE in ("stratified", "group")

if run_multiple_videos:
    RUN_DIRS = [f"../outputs/run_{n}_24DPE_n{n - 96:03d}" for n in range(97, 103)]
    tag = "24DPE_n001_to_n006"
else:
    RUN_DIRS = [r"../outputs/run_64_41DPE_n004"]
    tag = os.path.basename(os.path.normpath(RUN_DIRS[0]))

classif_root = "../outputs/classification"
os.makedirs(classif_root, exist_ok=True)
prefix = f"{tag}_run"
existing = [d for d in os.listdir(classif_root) if d.startswith(prefix)]
nums = []
for d in existing:
    rest = d.removeprefix(prefix)
    if rest.isdigit():
        nums.append(int(rest))
FIGURES_DIR = os.path.join(classif_root, f"{tag}_run{max(nums, default=0) + 1}")
os.makedirs(FIGURES_DIR, exist_ok=True)
print("Figures dir:", FIGURES_DIR)

## 2 — Load data and map genotypes

`map_vial_to_genotype` parses the filename to infer which vial corresponds
to which genotype (e.g. `..._hTDP43_WT-Het-Homo_...`).

In [ ]:
parts = []
for rd in RUN_DIRS:
    d = map_vial_to_genotype(rd)
    run_tag = os.path.basename(os.path.normpath(rd))
    d["run"] = run_tag
    d["run_dir"] = rd
    d["compact_id"] = run_tag + "::" + d["compact_id"].astype(str)
    parts.append(d)
df_raw = pd.concat(parts, ignore_index=True)
print(df_raw.shape, "| runs:", df_raw["run"].nunique())
print(df_raw["genotype"].value_counts())
df_raw.head()

## 3 — Extract behavioural features

Computes frame-level kinematics (velocity, acceleration, turning angle),
convex-hull area, and path tortuosity for each fly.

In [ ]:
df_feat = extract_behavioral_features(df_raw)
print(df_feat.shape)
df_feat[["compact_id", "frame", "velocity", "turning_angle", "area_covered", "tortuosity"]].head()

## 4 — Aggregate to per-fly features

In [ ]:
df_agg = aggregate_per_fly_features(df_feat, pause_threshold=1.0)

meta = (
    df_raw.drop_duplicates("compact_id")
    .set_index("compact_id")[["genotype", "run"]]
)
df_agg = df_agg.join(meta, on="compact_id").dropna(subset=["genotype"])

print(df_agg.shape)
df_agg.head()

In [ ]:
FEATURES = [
    "mean_velocity",
    "median_velocity",
    "pause_fraction",
    "total_distance_traveled",
    "tortuosity",
    "area_covered",
]

FEATURE_TITLES = {
    "mean_velocity":            "Mean velocity (px/s)",
    "median_velocity":          "Median velocity (px/s)",
    "pause_fraction":           "Pause fraction",
    "total_distance_traveled":  "Total distance traveled (px)",
    "tortuosity":               "Path tortuosity",
    "area_covered":             "Area covered (px^2)",
}

hover_data = ["compact_id", "run"]

In [ ]:
# Per-run sanity check: batch / lighting effects should not dwarf genotype
for feat in FEATURES:
    fig = px.box(
        df_agg, x="run", y=feat, color="genotype",
        points="all", hover_data=hover_data,
        title=f"{FEATURE_TITLES[feat]} — distribution per run",
    )
    fig.update_traces(jitter=0.3, marker=dict(size=6, opacity=0.75))
    fig.write_html(os.path.join(FIGURES_DIR, f"{feat}_per_run.html"))
    fig.show()

## 5 — Exploratory visualisation

Box plots for each feature, grouped by genotype (uses `FEATURES` / `hover_data` from above).

In [ ]:
plot_by_genotype(df_agg, FEATURES, FEATURE_TITLES, hover_data, outdir=FIGURES_DIR)

In [ ]:
plot_wt_vs_mutant(df_agg, FEATURES, FEATURE_TITLES, hover_data, outdir=FIGURES_DIR)

## 6 — Classification

Train LDA, Logistic Regression, and SVC classifiers.  
`CV_MODE == "group"` uses **GroupKFold** (no fly from a held-out video in training). Needs **≥2 runs**.  
`CV_MODE == "stratified"` pools all flies across videos (default sklearn splitter).

Figures go to `FIGURES_DIR` (auto-incremented under `outputs/classification/`).

In [ ]:
groups = df_agg["run"].values if CV_MODE == "group" else None
print(
    f"CV scheme: {CV_MODE}"
    + (f" ({df_agg['run'].nunique()} video groups, GroupKFold)" if groups is not None else " (stratified)"),
)

for model_name in ["lda", "logistic", "svc"]:
    for mode in ["multiclass", "binary"]:
        print(f"\n=== {model_name.upper()} [{mode}] ===")
        run_classifier(
            df=df_agg,
            outdir=FIGURES_DIR,
            model_name=model_name,
            classification_mode=mode,
            cv=5,
            plot_importance=True,
            groups=groups,
        )

## 7 — Report site (`classification_report.html`) + optional latent-space page

Build a navigable report with a sidebar entry page and modular section files (pooled + per-trial pages) to avoid huge single-file scrolling. If enabled, latent-space outputs are exported and linked into the same navigation hub.

In [ ]:
latent_rel = None
if RUN_LATENT_SPACE:
    latent = run_latent_space_analysis(df_raw)
    latent_dir = os.path.join(FIGURES_DIR, "latent_space")
    os.makedirs(latent_dir, exist_ok=True)

    latent["analysis1"]["umap_fig"].write_html(os.path.join(latent_dir, "umap_xy_kinematics.html"))
    latent["analysis2"]["umap_fig"].write_html(os.path.join(latent_dir, "umap_hist_kinematics.html"))
    latent["rf_importance_fig"].write_html(os.path.join(latent_dir, "rf_importance.html"))

    p1a = latent["analysis1"]["permanova"]["run_aware"]
    p1b = latent["analysis1"]["permanova"]["pooled"]
    p2a = latent["analysis2"]["permanova"]["run_aware"]
    p2b = latent["analysis2"]["permanova"]["pooled"]

    latent_page = os.path.join(FIGURES_DIR, "latent_space_report.html")
    with open(latent_page, "w", encoding="utf-8") as f:
        f.write("\n".join([
            "<!DOCTYPE html><html lang='en'><head><meta charset='utf-8'/><title>Latent-space report</title>",
            "<style>body{font-family:system-ui,sans-serif;margin:1rem 2rem;max-width:1200px;} .card{background:#f7f7f8;padding:0.6rem 0.8rem;border-radius:6px;margin:0.6rem 0;} iframe{width:100%;height:560px;border:1px solid #ddd;margin:0.5rem 0 1.2rem;}</style>",
            "</head><body>",
            "<h1>Latent-space report</h1>",
            f"<div class='card'><strong>Analysis 1 PERMANOVA (run-aware, primary)</strong> - method={p1a['method']}, pseudo-F={p1a['pseudo_f']:.4f}, p={p1a['p_value']:.4g}, R2={p1a['r2']:.4f}</div>",
            f"<div class='card'><strong>Analysis 1 PERMANOVA (pooled, exploratory)</strong> - method={p1b['method']}, pseudo-F={p1b['pseudo_f']:.4f}, p={p1b['p_value']:.4g}, R2={p1b['r2']:.4f}</div>",
            f"<div class='card'><strong>Analysis 2 PERMANOVA (run-aware, primary)</strong> - method={p2a['method']}, pseudo-F={p2a['pseudo_f']:.4f}, p={p2a['p_value']:.4g}, R2={p2a['r2']:.4f}</div>",
            f"<div class='card'><strong>Analysis 2 PERMANOVA (pooled, exploratory)</strong> - method={p2b['method']}, pseudo-F={p2b['pseudo_f']:.4f}, p={p2b['p_value']:.4g}, R2={p2b['r2']:.4f}</div>",
            "<h2>Fly trajectory and kinematic embedding (3D)</h2><iframe src='latent_space/umap_xy_kinematics.html'></iframe>",
            "<h2>Fly kinematic distribution embedding (3D)</h2><iframe src='latent_space/umap_hist_kinematics.html'></iframe>",
            "<h2>Random Forest importance of kinematic histogram bins</h2><iframe src='latent_space/rf_importance.html'></iframe>",
            "</body></html>",
        ]))
    latent_rel = "latent_space_report.html"

if WRITE_REPORT_SITE:
    _groups_report = df_agg["run"].values if CV_MODE == "group" else None
    report_entry = write_classification_report_site(
        df=df_agg,
        features=FEATURES,
        feature_titles=FEATURE_TITLES,
        hover_data=hover_data,
        out_dir=FIGURES_DIR,
        trial_column="run",
        report_title=f"{tag} — classification and latent space",
        pooled_cv=5,
        pooled_cv_groups=_groups_report,
        per_trial_cv=5,
        entry_filename="classification_report.html",
        latent_page_filename=latent_rel,
    )
    print("Wrote", os.path.abspath(report_entry))
else:
    print("WRITE_REPORT_SITE=False — skipped navigable report export.")

## Summary

Per-figure exports (HTML + PNG) are under `FIGURES_DIR`. The report entry page is `classification_report.html`, which links pooled/per-trial sections and (if enabled) latent-space outputs without forcing one giant single-page HTML.